In [ ]:
import numpy as np
import skimage as ski
import sys
from scipy import ndimage as ndi
from matplotlib import pyplot as plt
import struct


Program generiše tekstualne fajlove za simulaciju iz binarnih ulaza `parrots.bin`, `lena_128.bin` i `lena_512.bin`. 
Izlazni podaci se upisuju u `txt` fajlove (pixel po pixel) da bi mogli samo da ih ucitamo u tb.

In [ ]:
def bin_to_txt(bin_path, txt_path, has_header=True, dtype=np.int16):
    with open(bin_path, 'rb') as f:
        if has_header:
            print("Image: ", bin_path)
            
            image_shape = [0, 0]    
        
            data_bytes = f.read(2)
            image_shape[0] = struct.unpack('<H', data_bytes)[0]

            data_bytes = f.read(2)
            image_shape[1] = struct.unpack('<H', data_bytes)[0]

            print(f"Image shape: {image_shape}")

            print(f"Image height: {image_shape[0]}")
            print(f"Image width : {image_shape[1]}")

            # 2. Read a 2-byte unsigned short containing number of planes
            data_bytes = f.read(2)
            planes = struct.unpack('<H', data_bytes)[0]
            print(f"Number of image planes: {planes}")

            # 3. Read a 2-byte unsigned short containing pixel size in bytes
            data_bytes = f.read(2)
            pixel_size = struct.unpack('<H', data_bytes)[0]
            print(f"Pixel size in bytes: {pixel_size}\n")

        data = np.fromfile(f, dtype=dtype)

    with open(txt_path, 'w') as out:
        for val in data:
            out.write(f"{val}\n")

    return data

# Odavde kreće skripta za testiranje 

In [ ]:
def write_bin_img(file_name, img_out, type = 'uint8'):
    # Open the file in write-binary mode ('wb')
    with open(file_name, 'wb') as f:
        # 1. Write a two 2-byte unsigned short containing height and width of image
        f.write(np.uint16(img_out.shape[0]))    # imgH 2 bajta 
        f.write(np.uint16(img_out.shape[1]))    # imgW 2 bajta 

        # 2. Write a 2-byte unsigned short containing number of planes
        if (len(img_out.shape) == 2):
            f.write(np.uint16(1))
        elif (len(img_out.shape) == 3):
            f.write(np.uint16(img_out.shape[2]))
        else:
            return -1
        
        # 3. Write a 2-byte unsigned short containing pixel size in bytes followed by pixel data
        if (type == 'uint8'):
            f.write(np.uint16(1)) # 2 bita za velicinu 
            if (len(img_out.shape) == 2): # grayscale 
                f.write(np.uint8(img_out.flatten()))
            elif (len(img_out.shape) == 3):
                for p in range(img_out.shape[2]):
                    f.write(np.uint8(img_out[:,:,p].flatten()))
        elif (type == 'uint16'):
            f.write(np.uint16(2))
            if (len(img_out.shape) == 2):
                f.write(np.uint16(img_out.flatten()))
            elif (len(img_out.shape) == 3):
                for p in range(img_out.shape[2]):
                    f.write(np.uint16(img_out[:,:,p].flatten()))
        elif (type == 'uint32'):
            f.write(np.uint16(4))
            if (len(img_out.shape) == 2):
                f.write(np.uint32(img_out.flatten()))
            elif (len(img_out.shape) == 3):
                for p in range(img_out.shape[2]):
                    f.write(np.uint32(img_out[:,:,p].flatten()))
        else:
            return -1




In [ ]:
def read_bin_img(file_name):
    # Open the file in read-binary mode ('rb')
    with open(file_name, 'rb') as f:
        # 1. Read a two 2-byte unsigned short containing height and width of image
        image_shape = [0, 0]    
        
        data_bytes = f.read(2)
        image_shape[0] = struct.unpack('<H', data_bytes)[0]

        data_bytes = f.read(2)
        image_shape[1] = struct.unpack('<H', data_bytes)[0]

        print(f"Image shape: {image_shape}")

        print(f"Image height: {image_shape[0]}")
        print(f"Image width: {image_shape[1]}")

        # 2. Read a 2-byte unsigned short containing number of planes
        data_bytes = f.read(2)
        planes = struct.unpack('<H', data_bytes)[0]
        print(f"Number of image planes: {planes}")

        # 3. Read a 2-byte unsigned short containing pixel size in bytes
        data_bytes = f.read(2)
        pixel_size = struct.unpack('<H', data_bytes)[0]
        print(f"Pixel size in bytes: {pixel_size}")
    
        # 4. Initialize image matrix

        if (pixel_size == 1):
            image = np.zeros((image_shape[0], image_shape[1], planes), dtype=np.uint8)
        elif (pixel_size == 2):
            image = np.zeros((image_shape[0], image_shape[1], planes), dtype=np.uint16)
        elif (pixel_size == 4):
            image = np.zeros((image_shape[0], image_shape[1], planes), dtype=np.uint32)
        else:
            return -1
        
        # 5. Read pixel data

        for p in range(planes):
            for row in range(image_shape[0]):
                for col in range(image_shape[1]):
                    data_bytes = f.read(pixel_size)
                    if (pixel_size == 1):
                        image[row, col, p] = struct.unpack('<B', data_bytes)[0]
                    elif (pixel_size == 2):
                        image[row, col, p] = struct.unpack('<H', data_bytes)[0]
                    elif (pixel_size == 4):
                        image[row, col, p] = struct.unpack('<I', data_bytes)[0]
                    else:
                        return -1

        # 6. Reshape image to 2D or 3D array
        
        if (planes == 1):
            image = np.array(image).reshape((image_shape[0], image_shape[1]))
        else:
            image = np.array(image).reshape((image_shape[0], image_shape[1], planes))
        
        return image

In [ ]:
def plot_img(SIGNED_OUTPUT, img_bin): 
    COEFF_FRACTIONAL_BITS = 15
    SCALE_FACTOR_FRACTIONAL_BITS = 12

    SIGNED_OUT_FRACTIONAL_BITS = 7
    SIGNED_OUT_INTEGER_BITS    = 9

    UNSIGNED_OUT_BITS = 8

    plt.figure()
    if (SIGNED_OUTPUT):
        # 1. Kastuj u np.int16 da bi se ispravno protumačile negativne vrednosti iz komplementa dvojke
        img_signed = img_bin.astype(np.int16)
        
        # 2. Podeli sa 2**7 (128) jer izlaz iz C koda ima 7 frakcionih bita (fiksni zarez)
        img_float = img_signed / (2**SIGNED_OUT_FRACTIONAL_BITS)
        
        # 3. Skaliraj na opseg vrednosti za Q9.7 format pomoću formule iz dvs25_image_filter.ipynb
        img = np.clip(img_float, -2**(SIGNED_OUT_INTEGER_BITS-1), 2**(SIGNED_OUT_INTEGER_BITS-1) - 1/2**SIGNED_OUT_FRACTIONAL_BITS)
        img_normalized = (img + 2**(SIGNED_OUT_INTEGER_BITS-1)) / (2**SIGNED_OUT_INTEGER_BITS - 1/2**SIGNED_OUT_FRACTIONAL_BITS)
        plt.imshow(img_normalized, cmap='gray')
    else:
        # iseci sliku na opseg od 0 do 255 pri prikazu (za Mode 0)
        img_float = img_bin.astype(np.float64)
        img = np.clip(img_float, 0, 2**UNSIGNED_OUT_BITS - 1)
        plt.imshow(img.astype(np.uint8), cmap='gray');
    
    return img; 

# Prikaz slike hw 

In [ ]:
#### HW OUTPUT SLIKE ####
#img_bin = read_bin_img("hw_out.bin")

### 1. bypass ### 
#img_bypass_hw = read_bin_img("hw_128_bypass.bin")

### 2. sobel radius = 1, mode = 0 ### 
#img_sobel_512 = read_bin_img("sobel_hw_512.bin")

### 3. sobel radius = 1, mode = 1 ### 
#signed_sobel = read_bin_img("sobel_signed_h.bin")

### 4. box filter za lena 512 x 512 , radius = 2 , mode = 0 ### 
#box_filtered_img = read_bin_img("box_h.bin")

### 5. gauss obican za lena 128 x 128, radius = 4, mode = 0 ### 
#u_gauss = read_bin_img("gauss_128_h.bin")

### 6. log filter papagaji , radius = 0 , mode = 1 #### 
#log_parrots_hw = read_bin_img("parrots_h.bin")

### 7. sharpen filter lena 128 x 128 gauss baziran , radius = 3 ###
# gauss_sharpen = read_bin_img("sharpen_gauss_h.bin")

### 8. sharpen filter papagaji 512 x 768 box baziran, mode = 0, radius = 4 ###  
# box_parrots = read_bin_img("parrots_box_h.bin")

### 9. radijus 0, box = 0, 128 x 128 
# box_128_r0 = read_bin_img("hw_out.bin")


### 10. radijus = 2, sobel papagaji 
sobel_parrots = read_bin_img("sobel_parrots_h.bin")

## Za LoG filtar izlaz treba da bude označen

#hw_img = plot_img(SIGNED_OUTPUT = False, img_bin = img_bypass_hw)
#hw_img = plot_img(SIGNED_OUTPUT = False, img_bin = img_sobel_512)
#hw_img = plot_img(SIGNED_OUTPUT = False, img_bin = box_filtered_img)

#hw_img = plot_img(SIGNED_OUTPUT = True, img_bin = signed_sobel)
#hw_img = plot_img(SIGNED_OUTPUT = False, img_bin = u_gauss)

#hw_img = plot_img(SIGNED_OUTPUT = True, img_bin  = log_parrots_hw)
#hw_img = plot_img(SIGNED_OUTPUT = False, img_bin = gauss_sharpen)
#hw_img = plot_img(SIGNED_OUTPUT = False, img_bin = box_parrots)

#hw_img = plot_img(SIGNED_OUTPUT = False, img_bin = box_128_r0)
hw_img  = plot_img(SIGNED_OUTPUT = False, img_bin = sobel_parrots)

In [ ]:
### SW ### 

### 1.bypass ### 
#img_bypass_sw = read_bin_img("sw_128_bypass.bin")

### 2. papagaji log , radius = 0 ### 
log_parrots_sw = read_bin_img("parrots_s.bin")

### Prikaz slika ### 
sw_img = plot_img(SIGNED_OUTPUT = True, img_bin = log_parrots_sw)

#sw_img = plot_img(SIGNED_OUTPUT = True, img_bin = img_bypass_sw)

In [ ]:
#img_bin = read_bin_img("lena_sw128.bin")
#img_bin = read_bin_img('hw_img.bin')
img_bin = read_bin_img("hw_out.bin")

## Za LoG filtar izlaz treba da bude označen
plot_img(SIGNED_OUTPUT = False, img_bin = img_bin)

In [ ]:
bin_to_txt("parrots_sw.bin", "sw_img.txt")
bin_to_txt("parrots_hw.bin", "hw_img.txt")

In [ ]:
bin_to_txt("hw_out.bin", "last_wrong_hw_log.txt")
bin_to_txt("sw_ref.bin", "last_wrong_sw_log.txt")

# Prikaz slike sw

In [ ]:
img_bin = read_bin_img('sw_ref.bin')
#img_bin = read_bin_img('parrots_SW.bin')

COEFF_FRACTIONAL_BITS = 15
SCALE_FACTOR_FRACTIONAL_BITS = 12

SIGNED_OUT_FRACTIONAL_BITS = 7
SIGNED_OUT_INTEGER_BITS = 9

UNSIGNED_OUT_BITS = 8

SIGNED_OUTPUT = False #Za LoG filtar izlaz treba da bude označen

plot_img(SIGNED_OUTPUT = False, img_bin = img_bin)

## Definicija kooridinata i parametara za filtre(Python edition)

Smatra se da se u pythonu koeficijenti nalaze u sredini za filtriranje slike, u softveru pomereno u levi ugao jer tako prima hardver

In [ ]:
COEFF_FRACTIONAL_BITS = 15
SCALE_FACTOR_FRACTIONAL_BITS = 12

SIGNED_OUT_FRACTIONAL_BITS = 7
SIGNED_OUT_INTEGER_BITS = 9

UNSIGNED_OUT_BITS = 8

radius = 0

dirac = np.zeros((9, 9))
dirac[4, 4] = 1

dx = list(range(-4,5))
dy = list(range(-4,5))

# Meshgrid koordinate (X, Y) za matricu dimenzija 9x9 (radius=4):
#
#     -4 -3 -2 -1  0  1  2  3  4  (X vrednosti)
#  -4 [ .  .  .  .  .  .  .  .  . ]
#  -3 [ .  .  .  .  .  .  .  .  . ]
#  -2 [ .  .  .  .  .  .  .  .  . ]
#  -1 [ .  .  .  .  .  .  .  .  . ]
#   0 [ .  .  .  .  0  .  .  .  . ] (0,0) u centru
#   1 [ .  .  .  .  .  .  .  .  . ]
#   2 [ .  .  .  .  .  .  .  .  . ]
#   3 [ .  .  .  .  .  .  .  .  . ]
#   4 [ .  .  .  .  .  .  .  .  4 ] (4,4) u donjem desnom uglu
# (Y vrednosti)
dX, dY = np.meshgrid(dx, dy)

## Definicije filtara

In [ ]:
### Idealni filtar za usrednjavanje - box filtar ###

h_filter_box = (abs(dX) <= radius) & (abs(dY) <= radius) 
h_filter_box = h_filter_box / np.sum(h_filter_box.flatten())

####################################################


### Gausov filtar za usrednjavanje ##################

sigma = 1
h_filter_gauss = np.exp(-(np.square(dX) + np.square(dY))/(2*(sigma**2)))
h_filter_gauss[(abs(dX) > radius) | (abs(dY) > radius)] = 0
h_filter_gauss = h_filter_gauss / np.sum(h_filter_gauss.flatten())

#####################################################


### LoG (Laplacian of Gaussian) filtar ##############

sigma = 1
#h_filter_log = ndi.gaussian_laplace(dirac, sigma=sigma)
h_filter_log = -1/(np.pi*sigma**4)*(1 - (dX**2 + dY**2)/(2*sigma**2))*np.exp(-(np.square(dX) + np.square(dY))/(2*(sigma**2)))
h_filter_log[(abs(dX) > radius) | (abs(dY) > radius)] = 0

#####################################################

### Sharpen filter ##################################

k = 1

h_filter_sharpen_box = (1+k)*dirac - k*h_filter_box

h_filter_sharpen_gauss = (1+k)*dirac - k*h_filter_gauss

#####################################################

## Izbor filtra 

In [ ]:
h_filter = h_filter_log

In [ ]:
h_filter = h_filter_sharpen_gauss

# Kvantizacija 

In [ ]:
### COEFFICIENT QUANTIZATION #######################
title = "log 1 x 1" 

inv_scale_factor = 1

scale_factor = np.round((1 / inv_scale_factor) * 2**SCALE_FACTOR_FRACTIONAL_BITS)

print('\nScale factor: ' + str(scale_factor/2**SCALE_FACTOR_FRACTIONAL_BITS))
print('Scale factor INT VALUE: ' + str(scale_factor.astype(np.uint16)) + '\n')

h_filter_fxp = np.round(h_filter * 2**COEFF_FRACTIONAL_BITS * inv_scale_factor)
print(h_filter_fxp.astype(np.int16))

h_filter_fxp_scaled = h_filter_fxp * scale_factor / 2**SCALE_FACTOR_FRACTIONAL_BITS     

print('\nSum of fixed point scaled coeffs: ' + str(np.sum(h_filter_fxp_scaled/2**COEFF_FRACTIONAL_BITS)))
print('\nFixed point scaled error: ' + str(np.max(np.abs(h_filter_fxp_scaled/2**COEFF_FRACTIONAL_BITS - h_filter))) + '\n')

fig = plt.figure(figsize=(8,8), dpi=120)
ax = plt.axes(projection='3d')
ax.plot_surface(dX, dY, np.round(h_filter * 2**COEFF_FRACTIONAL_BITS * inv_scale_factor), cmap='rainbow'); ax.set_title(title);


In [ ]:
def plt_img_py_2d(SIGNED_OUTPUT, img, radius, h_filter_fxp_scaled):
    plt.figure()
    plt.imshow(img, cmap = 'gray');
    ref_img      = np.zeros((img.shape[0]-2*radius, img.shape[1]-2*radius)) # zauzmi izlazne dimenzije slike 
    img_out_full = ndi.correlate(img.astype(np.float64), h_filter_fxp_scaled) # filtriraj sa izabranim filtrom 
    ref_img      = img_out_full[radius:img.shape[0]-radius, radius:img.shape[1]-radius] / 2**COEFF_FRACTIONAL_BITS # iseci sliku od r do img - r 

    plt.figure()
    if (SIGNED_OUTPUT):
        # skaliraj na opseg vrednosti 
        ref_img = np.clip(ref_img, -2**(SIGNED_OUT_INTEGER_BITS-1), 2**(SIGNED_OUT_INTEGER_BITS-1) - 1/2**SIGNED_OUT_FRACTIONAL_BITS)
        plt.imshow((ref_img + 2**(SIGNED_OUT_INTEGER_BITS-1))/(2**SIGNED_OUT_INTEGER_BITS - 1/2**SIGNED_OUT_FRACTIONAL_BITS), cmap = 'gray');
    else:
        # iseci sliku na opseg od 0 do 255 pri prikazu 
        ref_img = np.clip(ref_img, 0, 2**UNSIGNED_OUT_BITS - 1)
        plt.imshow(ref_img.astype(np.uint8), cmap = 'gray');
        
    return ref_img
        
        
def plt_img_py_3d(SIGNED_OUTPUT, img, radius, h_filter_fxp_scaled):
    plt.figure();
    
    plt.imshow(img);
        
    #zauzmi prostor za isfiltriranu sliku  
    ref_img = np.zeros((img.shape[0]-2*radius, img.shape[1]-2*radius, 3))
    
    # reci da je originalna dimenzija slike sve nule 
    img_out_full = np.zeros(img.shape)
    
    # za svaku ravan lupi filtar izabrani 
    for p in range(0,3):
        img_out_full[:,:,p] = ndi.correlate(img[:,:,p].astype(np.float64), h_filter_fxp_scaled)
    
    #uzmi isecenu sliku onu bez ivica 
    ref_img = img_out_full[radius:img.shape[0]-radius, radius:img.shape[1]-radius, :] / 2**COEFF_FRACTIONAL_BITS

    plt.figure()
    if (SIGNED_OUTPUT):
        # skaliraj na opseg vrednosti 
        ref_img = np.clip(ref_img, -2**(SIGNED_OUT_INTEGER_BITS-1), 2**(SIGNED_OUT_INTEGER_BITS-1) - 1/2**SIGNED_OUT_FRACTIONAL_BITS)
        plt.imshow((ref_img + 2**(SIGNED_OUT_INTEGER_BITS-1))/(2**SIGNED_OUT_INTEGER_BITS - 1/2**SIGNED_OUT_FRACTIONAL_BITS));
    else:
        # zakucaj vrednosti na 0 do 255 
        ref_img = np.clip(ref_img, 0, 2**UNSIGNED_OUT_BITS - 1)
        plt.imshow(ref_img.astype(np.uint8));
        
    return ref_img; 

# Python filtriranje (uzima i scale u obzir) 

In [ ]:
img = read_bin_img('parrots.bin')
SIGNED_OUTPUT = True ## Za LoG filtar izlaz treba da bude označen

if (len(img.shape) == 2): ref_img = plt_img_py_2d(SIGNED_OUTPUT, img, radius , h_filter_fxp_scaled);
 
# rgb 
else: ref_img = plt_img_py_3d(SIGNED_OUTPUT, img, radius, h_filter_fxp_scaled)

# HW vs SW vs Python realizacija poredjenje vrednosti 

In [ ]:
### Poređenje SW referentnog izlaza, HW izlaza i Python filtriranja ###

print("SW output shape:    ", sw_img.shape)
print("HW output shape:    ", hw_img.shape)
print("Python output shape:", ref_img.shape)

if sw_img.shape != ref_img.shape: print("\n!!! SW_SHAPE /= PY_SHAPE \n")
if hw_img.shape != ref_img.shape: print("\n!!! HW_SHAPE /= PY_SHAPE \n")
 

In [ ]:
#### razlika sw / ref_python_img ####    
diff= sw_img.astype(np.float64) - ref_img.astype(np.float64)

tol = 1  # tolerancija +/-1 zbog zaokruzivanja/truncation razlike SW vs Python
num_mismatch = np.sum(np.abs(diff) > tol)

print(f"Piksela sa |razlika| > {tol}: {num_mismatch} / {diff.size} "
      f"({100*num_mismatch/diff.size:.3f}%)")


plt.figure()
plt.imshow(np.abs(diff), cmap='hot')
plt.colorbar()
plt.title(f"Apsolutna razlika SW vs Python")
plt.show()


In [ ]:
#### razlika hw / ref_python_img    ####
diff= hw_img.astype(np.float64) - ref_img.astype(np.float64)

tol = 1  # tolerancija +/-1 zbog zaokruzivanja/truncation razlike SW vs Python
num_mismatch = np.sum(np.abs(diff) > tol)

print(f"Piksela sa |razlika| > {tol}: {num_mismatch} / {diff.size} "
      f"({100*num_mismatch/diff.size:.3f}%)")


plt.figure()
plt.imshow(np.abs(diff), cmap='hot')
plt.colorbar()
plt.title(f"Apsolutna razlika HW vs Python ")
plt.show()

In [ ]:
#### razlika hw / sw img ####    
diff = hw_img.astype(np.float64) - sw_img.astype(np.float64)

tol = 1  # tolerancija +/-1 zbog zaokruzivanja/truncation razlike SW vs Python
num_mismatch = np.sum(np.abs(diff) > tol)

print(f"Piksela sa |razlika| > {tol}: {num_mismatch} / {diff.size} "
      f"({100*num_mismatch/diff.size:.3f}%)")


plt.figure()
plt.imshow(np.abs(diff), cmap='hot')
plt.colorbar()
plt.title(f"Apsolutna razlika SW vs HW")
plt.show()